<a href="https://colab.research.google.com/github/veronicaluzzi/Data-Science-Cohort-20/blob/main/Project-5/NLP_part3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Natural Language Processing



This project will give you practical experience using Natural Language Processing techniques. This project is in three parts:
- in part 1) you will use a dataset in a CSV file
- in part 2) you will use the Wikipedia API to directly access content
on Wikipedia.
- in part 3) you will make your notebook interactive


### Part 3)


Make an interactive notebook where a user can choose or enter a name and the notebook displays the 10 closest individuals.

In addition to presenting the project slides, at the end of the presentation each student will demonstrate their code using a famous person suggested by the other students that exists in the DBpedia set.


In [1]:
%%capture output
#install Wikipedia API
!pip3 install wikipedia-api

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

from ipywidgets import interact, widgets
import re
import time

import wikipediaapi

pd.options.display.max_columns = 100

import nltk
# nltk.download('omw-1.4')
nltk.download('punkt_tab')
# nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

Part 1 code below

In [3]:
url = "https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv"
df = pd.read_csv(url)
df


,URI,name,text
0,<http://dbpedia.org/resource/Digby_Morrell>,Digby Morrell,digby morrell born 10 october 1979 is a former...
1,<http://dbpedia.org/resource/Alfred_J._Lewy>,Alfred J. Lewy,alfred j lewy aka sandy lewy graduated from un...
2,<http://dbpedia.org/resource/Harpdog_Brown>,Harpdog Brown,harpdog brown is a singer and harmonica player...
3,<http://dbpedia.org/resource/Franz_Rottensteiner>,Franz Rottensteiner,franz rottensteiner born in waidmannsfeld lowe...
4,<http://dbpedia.org/resource/G-Enka>,G-Enka,henry krvits born 30 december 1974 in tallinn ...
...,...,...,...
42781,<http://dbpedia.org/resource/Motoaki_Takenouchi>,Motoaki Takenouchi,motoaki takenouchi born july 8 1967 saitama pr...
42782,<http://dbpedia.org/resource/Alan_Judge_(footb...,"Alan Judge (footballer, born 1960)",alan graham judge born 14 may 1960 is a retire...
42783,<http://dbpedia.org/resource/Eduardo_Lara>,Eduardo Lara,eduardo lara lozano born 4 september 1959 in c...
42784,<http://dbpedia.org/resource/Tatiana_Faberg%C3...,Tatiana Faberg%C3%A9,tatiana faberg is an author and faberg scholar...


In [6]:
# converts the text in the text and name columns into strings.
def fast_mask_name(row):
    text = str(row['text'])
    name = str(row['name'])

    # Split the name into individual tokens (e.g., "Gary", "Griffin")
    name_parts = name.split()

    # Replace the full name and its individual parts with empty space or a generic token
    # We use regex boundaries (\b) to ensure we don't accidentally mask parts of other words
    for part in name_parts:
        if len(part) > 2: # Avoid masking tiny initials like "A."
            text = re.sub(rf'\b{re.escape(part)}\b', '', text, flags=re.IGNORECASE)

    return text

print(f"Masking names across {df.index.size} rows...")
# use .apply() or look into 'pandarallel' / 'dask' if this takes more than a couple of minutes
df['cleaned_text'] = df.apply(fast_mask_name, axis=1)

Masking names across 42786 rows...


In [11]:
# Use .iloc[0] to grab the text of the person of interest (Gary Griffin)
# This looks at the 42630th row, and extracts whatever is in the 'text' column
griffin_full_text = df["cleaned_text"].iloc[42630]

# Create a TextBlob of his full text profile
full_blob = TextBlob(griffin_full_text)
full_blob

TextBlob("  born october 26 1951 in cincinnati ohio is an american musician best known for performing as a keyboardist and vocalist with the beach boys brian wilson jan and dean and the surf city allstarsgriffin grew up in cincinnati and attended miami university and the university of cincinnati college conservatory of music where he majored in piano and music theoryin 1977  moved to los angeles where he was hired as an organist by the beach boys joining them for the recording of their warner bros album miu  toured and recorded with jan and dean throughout the 80s and 90s and has appeared on several television shows most notably general hospital and full house in 2000  served as music director for the emmynominated television miniseries the beach boys an american family for abc he also appeared in two roles in the film which was produced by john stamos  has produced an eclectic roster of artists most notably micky dolenz of the monkees and political satirist harry shearer he also copro

In [12]:
# Initialize the CountVectorizer
# stop_words='english' removes common filler words like 'the', 'is', 'and'
vectorizer = CountVectorizer(stop_words="english")

In [13]:
# Transform the entire text column straight into a Bag of Words matrix
# it works on all rows simultaneously!
bow_alltext = vectorizer.fit_transform(df["cleaned_text"])

In [14]:
# Transform BoW to TF-IDF
tfidf_transformer = TfidfTransformer()
tfidf_matrix = tfidf_transformer.fit_transform(bow_alltext)

# Fit your KNN model on the matrix
knn = NearestNeighbors(n_neighbors=11, metric="cosine")
knn.fit(tfidf_matrix)

NearestNeighbors(metric='cosine', n_neighbors=11)

In [15]:
# 1.For this code to work, I need to run part 1 first. The code below is using the references df, knn, and code from cells above.
# 2. Use the @interact decorator to automatically generate the slider.
# When you move the slider, it selects an integer (row_index = 500). I choose a max = 1000 to be brief.
@interact(row_index=widgets.IntSlider(min=0, max=df.index.size, step=1, value=df.index.size//2, description='Row Index:'))
# 3. Create a function to gather all the information I need for the slider
def show_closest_by_row(row_index):
    # Query the KNN model using the selected slider index
    # (Matches  the code above: distances, indices = knn.kneighbors(...))
    distances, indices = knn.kneighbors(tfidf_matrix[row_index], n_neighbors=11)

    neighbor_indices = indices[0]
    neighbor_distances = distances[0]

    # Create a temporary DataFrame to process and display results
    results_df = df.iloc[neighbor_indices].copy()
    results_df['Cosine Distance'] = neighbor_distances

    # Remove the reference individual themselves so only the 10 closest show
    # results_df = results_df[results_df.index != row_index]

    # Clean up the dataframe so is human readable
    results_df.insert(0, 'Rank', range(0, 11))
    results_df = results_df[['Rank', 'name', 'Cosine Distance']]
    results_df.set_index('Rank', inplace=True)

    # Add an f string to make clear who the target person is

    print(f"Target Person at row {row_index}: {df.loc[row_index, 'name']}\n")
    display(results_df)

interactive(children=(IntSlider(value=21393, description='Row Index:', max=42786), Output()), _dom_classes=('w…